# Station Stacking v17 - KATL

Experimental notebook for `KATL`.

V17 is the importance-pruned experiment: only the KATL v16 fused features with `max_importance_mae_f >= 0.015`.


In [1]:
from pathlib import Path
import os
import sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*")
warnings.filterwarnings("ignore", message="Skipping features without any observed values.*")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing src/calibration/station_stacking.py")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ["WEATHER_RESEARCH_INCLUDE_DIRECT_NBM"] = "1"

STATION_ID = "KATL"
PROVIDERS = ("gfs", "hrrr", "nbm")
TIMING_MODE = "same_day_11am_live_safe"
TARGET_SOURCE = "iem_hourly"
FAST_MODE = False
OPTUNA_TRIALS = 30
STACK_OPTUNA_TRIALS = 30
OPTUNA_STARTUP_TRIALS = 15
STACK_OPTUNA_STARTUP_TRIALS = 15
OPTUNA_METRIC = "mae_f"
OPTUNA_VERBOSE = True
EXPORT_MODEL_WEIGHTS = True
OUTPUT_DIR = PROJECT_ROOT / "data" / "calibration" / "station_stacking_v17"
V15_OUTPUT_DIR = PROJECT_ROOT / "data" / "calibration" / "station_stacking_v15"
V16_OUTPUT_DIR = PROJECT_ROOT / "data" / "calibration" / "station_stacking_v16"
MODEL_VERSION = "station_high_regressor_v17_importance_015_stack"

PROJECT_ROOT


WindowsPath('D:/dev/weather-research')

In [2]:
import numpy as np
import pandas as pd

from scripts.run_station_stacking_v17 import write_reference_comparisons
from src.export_station_stacking_v2_models import export_station_model_weights
from src.calibration.station_stacking import (
    StationStackingConfig,
    V17_ADDITIONAL_FEATURE_COLUMNS,
    V17_DROPPED_FEATURE_COLUMNS,
    V17_IMPORTANCE_015_FEATURE_COLUMNS,
    YEAR_SPLIT_EXPANDING_FOLDS,
    missing_model_dependencies,
    provider_availability,
    run_station_year_split_experiment,
)


## V17 Contract

`feature_version="v17_importance_015"` uses only the 17-feature allowlist from the KATL v16 fused importance summary. The single v13 weather interaction is kept only when train-year coverage passes.


In [3]:
fold_spec = pd.DataFrame(
    [
        {
            "fold": fold.name,
            "train_start_year": fold.train_start_year,
            "train_end_year": fold.train_end_year,
            "validation_year": fold.validation_year,
        }
        for fold in YEAR_SPLIT_EXPANDING_FOLDS
    ]
)

{
    "folds": fold_spec,
    "v17_feature_count": len(V17_IMPORTANCE_015_FEATURE_COLUMNS),
    "v17_features": V17_IMPORTANCE_015_FEATURE_COLUMNS,
    "v17_coverage_gated_features": V17_ADDITIONAL_FEATURE_COLUMNS,
    "v17_dropped_features": sorted(V17_DROPPED_FEATURE_COLUMNS),
}


{'folds':                      fold  train_start_year  train_end_year  validation_year
 0  fold_2021_2023_to_2024              2021            2023             2024
 1  fold_2021_2024_to_2025              2021            2024             2025,
 'v17_feature_count': 17,
 'v17_features': ['nbm_high_minus_observed_high_temp_f',
  'v8_provider_median_remaining_from_high_so_far_f',
  'observed_high_so_far_change_since_9am_f',
  'nbm_high_minus_observed_temp_f',
  'v3_remaining_warmup_per_spread_f',
  'v8_provider_max_remaining_from_high_so_far_f',
  'v8_provider_mean_remaining_vs_month_normal_f',
  'hrrr_high_minus_observed_high_temp_f',
  'observed_high_temp_minus_temp_at_as_of_f',
  'gfs_high_minus_observed_high_temp_f',
  'observed_cloud_cover_at_as_of',
  'v3_remaining_warmup_from_high_so_far_f',
  'v3_humidity_remaining_warmup_interaction',
  'v13_forecast_temp_bias_remaining_warmup_interaction',
  'gfs_high_minus_observed_temp_f',
  'hrrr_rolling_bias_30d_f',
  'observed_temp_change_l

## Data Availability


In [4]:
availability = provider_availability(
    PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
)

availability.loc[availability["station_id"].eq(STATION_ID)]


,station_id,provider,row_count,first_contract_date,last_contract_date
0,KATL,gfs,1998,2021-01-01,2026-06-21
1,KATL,hrrr,1998,2021-01-01,2026-06-21
2,KATL,nbm,1997,2021-01-01,2026-06-21


## Run Importance-Pruned Model


In [5]:
missing_packages = missing_model_dependencies()
if missing_packages:
    raise ImportError(
        "Missing station-stacking ML packages: "
        + ", ".join(missing_packages)
        + ". Install them with: python -m pip install -r requirements.txt"
    )

config = StationStackingConfig(
    station_id=STATION_ID,
    project_root=PROJECT_ROOT,
    timing_mode=TIMING_MODE,
    providers=PROVIDERS,
    fast_mode=FAST_MODE,
    optuna_trials=OPTUNA_TRIALS,
    stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_startup_trials=OPTUNA_STARTUP_TRIALS,
    stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS,
    optuna_metric=OPTUNA_METRIC,
    optuna_verbose=OPTUNA_VERBOSE,
    feature_version="v17_importance_015",
    target_mode="remaining_warmup",
    target_source=TARGET_SOURCE,
    base_model_methods=("xgboost", "lightgbm", "catboost"),
    stack_enabled=True,
    hyperparameter_space="wide",
    year_split_folds=YEAR_SPLIT_EXPANDING_FOLDS,
    year_split_test_train_years=(2021, 2025),
    year_split_test_year=2026,
    output_dir=OUTPUT_DIR / "importance_015",
    climatology_normals_path=PROJECT_ROOT / "data" / "calibration" / "station_stacking_v9" / "station_rolling_10y_daily_high_normals.csv",
)

config.resolved_optuna_storage_path()


WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v17/importance_015/KATL_optuna.sqlite3')

In [6]:
result = run_station_year_split_experiment(config)
result.scoreboard


D:\dev\weather-research\src\calibration\station_stacking.py:2711: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{prefix}_{feature_name}_abs_diff_f"] = (left_values - right_values).abs()
D:\dev\weather-research\src\calibration\station_stacking.py:2710: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"{prefix}_{feature_name}_diff_f"] = left_values - right_values
D:\dev\weather-research\src\calibration\station_stacking.py:2711: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fram

,period,method,count,mae_f,rmse_f
0,validation_2024_2025,xgboost,728,1.778845,2.510110
1,validation_2024_2025,lightgbm,728,1.718731,2.427563
2,validation_2024_2025,catboost,728,1.761090,2.454563
3,validation_2024_2025,provider_mean,728,2.832206,4.006255
4,validation_2024_2025,provider_median,728,2.782240,3.911262
5,validation_2024_2025,nbm_raw,728,2.792647,3.816647
6,validation_2024_2025,hrrr_raw,728,3.556181,4.953880
7,validation_2024_2025,gfs_raw,728,3.254687,4.459385
8,test_2026,xgboost,170,1.721887,2.362276
9,test_2026,lightgbm,170,1.604937,2.215045


In [7]:
if EXPORT_MODEL_WEIGHTS:
    exported_weights = export_station_model_weights(
        project_root=PROJECT_ROOT,
        station_id=STATION_ID,
        artifact_dir=config.resolved_output_dir(),
        model_version=MODEL_VERSION,
        timing_mode=config.timing_mode,
        providers=tuple(config.providers),
        feature_version=config.effective_feature_version,
        optuna_metric=config.effective_optuna_metric,
        target_mode=config.effective_target_mode,
        target_source=config.effective_target_source,
        base_model_methods=tuple(config.effective_base_model_methods),
        stack_enabled=config.stack_enabled,
        source_pipeline="notebooks/station_stacking_v17",
    )
    display((exported_weights.bundle_path, exported_weights.manifest_path))

write_reference_comparisons(OUTPUT_DIR, V15_OUTPUT_DIR, V16_OUTPUT_DIR, STATION_ID)


(WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v17/importance_015/model_weights/KATL_station_high_regressor_v17_importance_015_stack.joblib'),
 WindowsPath('D:/dev/weather-research/data/calibration/station_stacking_v17/importance_015/model_weights/KATL_station_high_regressor_v17_importance_015_stack.json'))

Wrote KATL v17 reference comparison: D:\dev\weather-research\data\calibration\station_stacking_v17\KATL_v17_importance_015_vs_references_common_date_comparison.csv


## Reference Comparison


In [8]:
comparison_path = OUTPUT_DIR / f"{STATION_ID}_v17_importance_015_vs_references_common_date_comparison.csv"
comparison = pd.read_csv(comparison_path) if comparison_path.exists() else pd.DataFrame()
comparison.sort_values(["method", "delta_mae_f", "reference"]) if not comparison.empty else comparison


,station_id,reference,comparison,feature_version,model_version,method,common_date_count,reference_mae_f,v17_mae_f,delta_mae_f,reference_rmse_f,v17_rmse_f,delta_rmse_f,v17_better_days,reference_better_days,tied_days,actual_mismatch_count,first_common_date,last_common_date
27,KATL,v16_fused,v17_importance_015,v17_importance_015,station_high_regressor_v17_importance_015_stack,catboost,170,1.664224,1.664076,-0.000148,2.299231,2.273572,-0.025659,84,86,0,0,2026-01-01,2026-06-21
18,KATL,v15_precip_cloud,v17_importance_015,v17_importance_015,station_high_regressor_v17_importance_015_stack,catboost,170,1.623187,1.664076,0.040889,2.292603,2.273572,-0.019031,72,98,0,0,2026-01-01,2026-06-21
0,KATL,v15_base,v17_importance_015,v17_importance_015,station_high_regressor_v17_importance_015_stack,catboost,170,1.582195,1.664076,0.081880,2.211646,2.273572,0.061926,75,95,0,0,2026-01-01,2026-06-21
9,KATL,v15_forecast_temp_at_as_of,v17_importance_015,v17_importance_015,station_high_regressor_v17_importance_015_stack,catboost,170,1.487267,1.664076,0.176809,2.134523,2.273572,0.139049,68,102,0,0,2026-01-01,2026-06-21
1,KATL,v15_base,v17_importance_015,v17_importance_015,station_high_regressor_v17_importance_015_stack,gfs_raw,170,3.328938,3.328938,0.000000,4.532080,4.532080,0.000000,0,0,170,0,2026-01-01,2026-06-21
10,KATL,v15_forecast_temp_at_as_of,v17_importance_015,v17_importance_015,station_high_regressor_v17_importance_015_stack,gfs_raw,170,3.328938,3.328938,0.000000,4.532080,4.532080,0.000000,0,0,170,0,2026-01-01,2026-06-21
19,KATL,v15_precip_cloud,v17_importance_015,v17_importance_015,station_high_regressor_v17_importance_015_stack,gfs_raw,170,3.328938,3.328938,0.000000,4.532080,4.532080,0.000000,0,0,170,0,2026-01-01,2026-06-21
28,KATL,v16_fused,v17_importance_015,v17_importance_015,station_high_regressor_v17_importance_015_stack,gfs_raw,170,3.328938,3.328938,0.000000,4.532080,4.532080,0.000000,0,0,170,0,2026-01-01,2026-06-21
2,KATL,v15_base,v17_importance_015,v17_importance_015,station_high_regressor_v17_importance_015_stack,hrrr_raw,170,3.216669,3.216669,0.000000,4.676677,4.676677,0.000000,0,0,170,0,2026-01-01,2026-06-21
11,KATL,v15_forecast_temp_at_as_of,v17_importance_015,v17_importance_015,station_high_regressor_v17_importance_015_stack,hrrr_raw,170,3.216669,3.216669,0.000000,4.676677,4.676677,0.000000,0,0,170,0,2026-01-01,2026-06-21


## Selected Feature Audit


In [9]:
expected = set(V17_IMPORTANCE_015_FEATURE_COLUMNS)
selected = set(result.feature_columns["feature"].astype(str))
{
    "selected_feature_count": len(selected),
    "expected_feature_count": len(expected),
    "selected_features": sorted(selected),
    "missing_expected": sorted(expected - selected),
    "unexpected_selected": sorted(selected - expected),
    "coverage_gated_selected": sorted(selected & set(V17_ADDITIONAL_FEATURE_COLUMNS)),
}


{'selected_feature_count': 17,
 'expected_feature_count': 17,
 'selected_features': ['gfs_high_minus_observed_high_temp_f',
  'gfs_high_minus_observed_temp_f',
  'hrrr_high_minus_observed_high_temp_f',
  'hrrr_rolling_bias_30d_f',
  'nbm_high_minus_observed_high_temp_f',
  'nbm_high_minus_observed_temp_f',
  'observed_cloud_cover_at_as_of',
  'observed_high_so_far_change_since_9am_f',
  'observed_high_temp_minus_temp_at_as_of_f',
  'observed_temp_change_last_3h_f',
  'v13_forecast_temp_bias_remaining_warmup_interaction',
  'v3_humidity_remaining_warmup_interaction',
  'v3_remaining_warmup_from_high_so_far_f',
  'v3_remaining_warmup_per_spread_f',
  'v8_provider_max_remaining_from_high_so_far_f',
  'v8_provider_mean_remaining_vs_month_normal_f',
  'v8_provider_median_remaining_from_high_so_far_f'],
 'missing_expected': [],
 'unexpected_selected': [],
 'coverage_gated_selected': ['v13_forecast_temp_bias_remaining_warmup_interaction']}

## Accidental Weather Sprawl Check


In [10]:
raw_weather_tokens = (
    "cloud",
    "ceiling",
    "dewpoint",
    "forecast_temp_at_as_of",
    "humidity",
    "precip",
    "pressure",
    "shortwave",
    "visibility",
    "wind_",
)
selected_features = result.feature_columns["feature"].astype(str)
accidental_weather_selected = pd.DataFrame(
    [
        {"feature": feature}
        for feature in selected_features
        if (
            feature.startswith(("gfs_", "hrrr_", "nbm_", "v13_", "v8_"))
            and any(token in feature for token in raw_weather_tokens)
            and feature not in set(V17_IMPORTANCE_015_FEATURE_COLUMNS)
        )
    ]
)

accidental_weather_selected


""


## Feature Importance


In [12]:
importance_path = config.resolved_output_dir() / f"{STATION_ID}_year_split_feature_importance.csv"
importance = pd.read_csv(importance_path) if importance_path.exists() else pd.DataFrame()
importance.sort_values(["method", "importance_mae_f"], ascending=[True, False]) if not importance.empty else importance


KeyError: 'importance_mae_f'

## Rounded Within 1F Accuracy


In [13]:
preds = pd.concat(
    [
        result.validation_predictions.assign(period="validation_2024_2025"),
        result.test_predictions.assign(period="oof_2026"),
    ],
    ignore_index=True,
)
predicted_high = pd.to_numeric(preds["predicted_high_f"], errors="coerce")
preds["predicted_high_rounded_f"] = np.floor(predicted_high + 0.5)
preds["within_1f_after_round"] = (
    pd.to_numeric(preds["actual_high_f"], errors="coerce") - preds["predicted_high_rounded_f"]
).abs().le(1)

within_1f_accuracy_by_period = (
    preds
    .dropna(subset=["actual_high_f", "predicted_high_rounded_f"])
    .groupby(["period", "method"], as_index=False)
    .agg(
        count=("within_1f_after_round", "size"),
        within_1f_count=("within_1f_after_round", "sum"),
        within_1f_accuracy_pct=("within_1f_after_round", lambda x: x.mean() * 100),
    )
    .sort_values(["period", "within_1f_accuracy_pct"], ascending=[True, False])
)

within_1f_accuracy_by_period


,period,method,count,within_1f_count,within_1f_accuracy_pct
7,oof_2026,ridge_stack,170,105,61.764706
3,oof_2026,lightgbm,170,101,59.411765
8,oof_2026,xgboost,170,99,58.235294
0,oof_2026,catboost,170,96,56.470588
4,oof_2026,nbm_raw,170,68,40.000000
5,oof_2026,provider_mean,170,63,37.058824
6,oof_2026,provider_median,170,63,37.058824
2,oof_2026,hrrr_raw,170,56,32.941176
1,oof_2026,gfs_raw,170,47,27.647059
12,validation_2024_2025,lightgbm,728,411,56.456044


## Bracket Metrics


In [14]:
result.bracket_metrics


,method,count,mae_f,rmse_f,bracket_accuracy_pct
0,xgboost,170,1.721887,2.362276,38.235294
1,lightgbm,170,1.604937,2.215045,40.588235
2,catboost,170,1.664076,2.273572,38.823529
3,ridge_stack,170,1.585664,2.201156,40.0
4,provider_mean,170,2.786086,4.048184,25.294118
5,provider_median,170,2.723802,3.926838,27.647059
6,nbm_raw,170,2.614555,3.750025,28.823529
7,hrrr_raw,170,3.216669,4.676677,24.705882
8,gfs_raw,170,3.328938,4.532080,20.588235
